# **2. Paraphrase Generation**

In [ ]:
!pip install -q transformers sentencepiece pandas torch

In [ ]:
import os
import re
import pandas as pd
import numpy as np
import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
from google.colab import drive

In [ ]:
drive.mount('/content/drive')

BASE_PATH = "/content/drive/MyDrive/commonsenseqa_paraphrase_evaluation"
print("Loading from:", BASE_PATH)

In [ ]:
def save_df(df, filename):
    path = os.path.join(BASE_PATH, filename)
    df.to_csv(path, index=False)
    print(f"Saved: {path}")

In [ ]:
# Load cleaned dataset from Notebook 01
filtered_df = pd.read_csv("/content/drive/MyDrive/commonsenseqa_paraphrase_evaluation/filtered_df.csv")

print(filtered_df.shape)
filtered_df.head()

### Helper Functions

In [ ]:
# Fix any messing formatting - unnecessary spacing
def clean_text(s: str) -> str:
    s = re.sub(r"\s+", " ", s).strip()
    s = s.replace(" ?", "?").replace(" ,", ",").replace(" .", ".")
    s = s.replace(" !", "!").replace(" ;", ";").replace(" :", ":")
    return s

# Check for duplicates of paraphrases
def normalize_for_dedup(s: str) -> str:
    s = clean_text(s).lower()
    s = re.sub(r"[^a-z0-9\s?]", "", s)
    s = re.sub(r"\s+", " ", s).strip()
    return s

# Check for valid Paraphrases
def soft_is_valid_paraphrase(candidate: str, original: str) -> bool:
    c = clean_text(candidate)
    o = clean_text(original)

    if len(c) < 10:
        return False
    if "?" not in c:
        return False
    if normalize_for_dedup(c) == normalize_for_dedup(o):
        return False

    bad_markers = ["<extra_id_", "</s>", "<pad>"]
    if any(marker in c for marker in bad_markers):
        return False

    return True

### Load T5 Paraphrase Model

In [ ]:
model_name = "Vamsi/T5_Paraphrase_Paws" # Paws T5 Model

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)
model.eval()

print("Using device:", device)

### Paraphrase Levels

In [ ]:
N_PER_LEVEL = 3 # Number of Questions per level

# Temperature: Controls randomness of word choice
# Top-p: Controls how many likely words the model can pick from
# Top-k: Limits choices to the top k most likely next words
# num_return_sequences: generate paraphrase candidates

LEVEL_CONFIG = {
    "light": {
        "temperature": 0.7,
        "top_p": 0.85,
        "top_k": 80,
        "num_return_sequences": 20
    },
    "medium": {
        "temperature": 0.95,
        "top_p": 0.90,
        "top_k": 120,
        "num_return_sequences": 20
    },
    "strong": {
        "temperature": 1.2,
        "top_p": 0.95,
        "top_k": 200,
        "num_return_sequences": 20
    }
}

LEVEL_CONFIG

### Paraphrase Generation

In [ ]:
def generate_level_paraphrases(
    question: str,
    temperature: float,
    top_p: float,
    top_k: int,
    num_return_sequences: int,
    max_input_length: int = 128,
    max_output_length: int = 128
):
    text = "paraphrase: " + question + " </s>" # Model Prompt

    # Tokenize Text
    encoding = tokenizer(
        text,
        padding=True,
        truncation=True,
        max_length=max_input_length,
        return_tensors="pt"
    )
    # Generate Paraphrases
    input_ids = encoding["input_ids"].to(device)
    attention_mask = encoding["attention_mask"].to(device)

    with torch.no_grad():
        outputs = model.generate(
            input_ids=input_ids,
            attention_mask=attention_mask,
            max_length=max_output_length,
            do_sample=True, # Allows Randomness
            temperature=temperature,
            top_p=top_p,
            top_k=top_k,
            early_stopping=True,
            num_return_sequences=num_return_sequences,
            repetition_penalty=1.1
        )
    # Store Results
    paraphrases = []
    seen = set()

    # Convert back to text
    for output in outputs:
        line = tokenizer.decode(
            output,
            skip_special_tokens=True,
            clean_up_tokenization_spaces=True
        )
        line = clean_text(line)

        # Filter out bad paraphrases: Too short, not questions, same as original
        if soft_is_valid_paraphrase(line, question):
            key = normalize_for_dedup(line)
            # Remove Duplicates
            if key not in seen:
                paraphrases.append(line)
                seen.add(key)

    return paraphrases


In [ ]:
# Organize paraphrases by level
def generate_all_levels(question: str, n_per_level: int = 3):
    results = {}

    for level_name, config in LEVEL_CONFIG.items():
        paras = generate_level_paraphrases(
            question=question,
            temperature=config["temperature"],
            top_p=config["top_p"],
            top_k=config["top_k"],
            num_return_sequences=config["num_return_sequences"]
        )

        paras = paras[:n_per_level]
        while len(paras) < n_per_level:
            paras.append("")

        results[level_name] = paras

    return results


### Dataframe of Paraphrase Levels

In [ ]:
# Organize dataframe by paraphrases
rows = []

for _, row in filtered_df.iterrows():
    question = row["question"]
    levels = generate_all_levels(question, n_per_level=N_PER_LEVEL)

    rows.append({
        "id": row["id"],
        "question": question,
        "choice_A": row["choice_A"],
        "choice_B": row["choice_B"],
        "choice_C": row["choice_C"],
        "choice_D": row["choice_D"],
        "choice_E": row["choice_E"],
        "answerKey": row["answerKey"],
        "light_1": levels["light"][0],
        "light_2": levels["light"][1],
        "light_3": levels["light"][2],
        "medium_1": levels["medium"][0],
        "medium_2": levels["medium"][1],
        "medium_3": levels["medium"][2],
        "strong_1": levels["strong"][0],
        "strong_2": levels["strong"][1],
        "strong_3": levels["strong"][2],
    })

paraphrase_df = pd.DataFrame(rows)
paraphrase_df.head()

# save_df_to_drive(paraphrase_df, "paraphrase_df.csv") --------- OPTIONAL SAVE TO GOOGLE DRIVE


### Dataframe For Zero-Shot Model Inference

In [ ]:
def wide_to_long_paraphrase_df(paraphrase_df):
    rows = []

    for _, row in paraphrase_df.iterrows():
        choices = {
            "A": row["choice_A"],
            "B": row["choice_B"],
            "C": row["choice_C"],
            "D": row["choice_D"],
            "E": row["choice_E"],
        }

        # original question
        rows.append({
            "id": row["id"],
            "variant_name": "original",
            "paraphrase_level": "original",
            "question_text": row["question"],
            "choices": choices,
            "answerKey": row["answerKey"]
        })

        # paraphrases
        for level in ["light", "medium", "strong"]:
            for i in range(1, 4):
                col = f"{level}_{i}"
                q_text = row[col]

                if isinstance(q_text, str) and q_text.strip() != "":
                    rows.append({
                        "id": row["id"],
                        "variant_name": col,
                        "paraphrase_level": level,
                        "question_text": q_text,
                        "choices": choices,
                        "answerKey": row["answerKey"]
                    })

    return pd.DataFrame(rows)

eval_df = wide_to_long_paraphrase_df(paraphrase_df)
print(eval_df.shape)
eval_df.head()

# save_df_to_drive(eval_df, "eval_df_long.csv") --------- OPTIONAL SAVE TO GOOGLE DRIVE